In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Converted from Jupyter Notebook: notebook.ipynb
Conversion Date: 2025-11-22T23:12:37.983Z

Embedding Extraction for Trained Multi-Modal Autoencoder


"""

# ======================
# Imports
# ======================

import os
import re
import sys
from pathlib import Path
from tempfile import NamedTemporaryFile

WORKING_DIR = Path.cwd()
if (WORKING_DIR / "code" / "00_training").is_dir():
    PROJECT_ROOT = WORKING_DIR
elif WORKING_DIR.name == "00_training" and WORKING_DIR.parent.name == "code":
    PROJECT_ROOT = WORKING_DIR.parents[1]
else:
    raise RuntimeError("Run this notebook from the repository root or code/00_training.")
TRAINING_DIR = PROJECT_ROOT / "code" / "00_training"
sys.path.insert(0, str(TRAINING_DIR))

import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader

from multimodal_autoencoder import MultiModalAutoencoder
from voxel_dataset import VoxelDataset
from tqdm import tqdm

# ======================
# ======================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

###############################################################################
METRIC_SCALE = 10.0
print(f"Using METRIC_SCALE = {METRIC_SCALE}")

QTY_FIXED_VALUE = 0.0
print(f"Embedding will use FIXED LogScaled_Quantity = {QTY_FIXED_VALUE} (qty-invariant embedding)")

# ======================
# DF helpers
# ======================

def expand_df_by_suppliers(df: pd.DataFrame) -> pd.DataFrame:
    """
    """
    df = df.copy()
    df['Supplier'] = df['Supplier'].astype(str)
    df['SupplierList'] = df['Supplier'].apply(lambda x: x.split(','))
    expanded_rows = []
    for _, row in df.iterrows():
        for supplier in row['SupplierList']:
            new_row = row.copy()
            new_row['Supplier'] = int(supplier.strip())
            expanded_rows.append(new_row)
    expanded_df = pd.DataFrame(expanded_rows)
    expanded_df.drop(columns=['SupplierList'], inplace=True)
    return expanded_df


def load_df(csv_path: str, voxel_dir: str, is_test: bool = False) -> pd.DataFrame:
    """
    """
    df = pd.read_csv(csv_path)
    if not os.path.isdir(voxel_dir):
        raise FileNotFoundError(f"BINVOX directory not found: {voxel_dir}")
    existing_binvox = {f for f in os.listdir(voxel_dir) if f.endswith(".binvox")}

    if 'FileName' in df.columns:
        name_col = 'FileName'
    elif 'filename' in df.columns:
        name_col = 'filename'
    else:
        raise KeyError("CSV must contain 'FileName' or 'filename'.")

    has_voxel = df[name_col].astype(str).map(
        lambda name: os.path.basename(name) in existing_binvox
    )
    df = df[has_voxel].reset_index(drop=True)
    if df.empty:
        raise RuntimeError(f"No CSV rows match BINVOX files in: {voxel_dir}")

    if 'Supplier' not in df.columns:
        raise KeyError("CSV must contain 'Supplier'.")

    df['Supplier'] = df['Supplier'].astype(str)
    df['SupplierList'] = df['Supplier'].apply(
        lambda x: [int(s.strip()) for s in x.split(',') if s.strip() != ""]
    )

    if not is_test:
        df = expand_df_by_suppliers(df)

    return df

# ======================
# Embedding extraction helper
# ======================

def embed_and_save(model,
                   df: pd.DataFrame,
                   voxel_dir: str,
                   device,
                   save_path: str,
                   is_test: bool = False,
                   add_noise_std: float = 0.0):
    """
    """
    df = df.copy()

    if 'filename' not in df.columns:
        if 'FileName' in df.columns:
            df.rename(columns={'FileName': 'filename'}, inplace=True)
        else:
            raise KeyError("DataFrame must contain 'filename' or 'FileName'.")

    phase_name = "test" if is_test else "train"
    output_dir = os.path.dirname(os.path.abspath(save_path))
    os.makedirs(output_dir, exist_ok=True)
    temp_file = NamedTemporaryFile(
        mode="w", suffix=".csv", prefix=f".{phase_name}_",
        dir=output_dir, delete=False
    )
    temp_csv_path = temp_file.name
    temp_file.close()
    df.to_csv(temp_csv_path, index=False)

    batch_size = 8
    try:
        dataset = VoxelDataset(csv_file=temp_csv_path, voxel_dir=voxel_dir)
    finally:
        if os.path.exists(temp_csv_path):
            os.remove(temp_csv_path)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=8,
        pin_memory=(device.type == "cuda")
    )

    total_n = len(df)
    print(f"[DEBUG] embed_and_save start: phase={phase_name}, N={total_n}")

    model.eval()
    embeddings, filenames, suppliers = [], [], []

    with torch.no_grad():
        for batch_idx, voxel in enumerate(loader):
            start = batch_idx * batch_size
            end = min(start + voxel.size(0), total_n)
            batch_df = df.iloc[start:end]

            if batch_idx % 100 == 0:
                print(f"[DEBUG] {phase_name}: processed {start}/{total_n}")

            voxel = voxel.to(device).float()
            B = voxel.size(0)

            t = torch.tensor(
                batch_df['LogScaled_Time'].values,
                dtype=torch.float32,
                device=device
            ).view(B, 1) * METRIC_SCALE

            c = torch.tensor(
                batch_df['LogScaled_Cost'].values,
                dtype=torch.float32,
                device=device
            ).view(B, 1) * METRIC_SCALE

            q = torch.full(
                (B, 1),
                fill_value=QTY_FIXED_VALUE * METRIC_SCALE,
                dtype=torch.float32,
                device=device
            )

            tol = torch.tensor(
                batch_df['LogScaled_Tolerance'].values,
                dtype=torch.float32,
                device=device
            ).view(B, 1) * METRIC_SCALE

            m = torch.tensor(
                batch_df[
                    [
                        'LogScaled_Density',
                        'LogScaled_Service_Temperature',
                        'LogScaled_Ultimate_Tensile'
                    ]
                ].values,
                dtype=torch.float32,
                device=device
            ) * METRIC_SCALE  # [B,3]

            # forward
            _, _, z = model(voxel, t, c, q, tol, m)   # z: [B, latent_dim]
            z_cpu = z.cpu().numpy()

            for j in range(B):
                row = batch_df.iloc[j]
                filename = row['filename']

                emb = z_cpu[j]
                if (not is_test) and add_noise_std and add_noise_std > 0.0:
                    emb = emb + np.random.normal(0, add_noise_std, size=emb.shape)

                embeddings.append(emb.tolist())
                filenames.append(filename)

                if is_test:
                    suppliers.append(row['SupplierList'])
                else:
                    suppliers.append(int(row['Supplier']))

    print(f"[DEBUG] embed_and_save done: phase={phase_name}, N={total_n}")

    df_out = pd.DataFrame({
        "filename": filenames,
        "supplier": suppliers,
        "embedding": embeddings
    })
    df_out.to_csv(save_path, index=False)
    print(f"[✓] Saved embeddings to: {save_path}")

# ======================
# ======================

def parse_epoch_from_filename(fname: str):
    """
    'multimodal_autoencoder_epoch_050.pth' -> 50
    """
    m = re.search(r'(\d+)(?=\.pth$)', os.path.basename(fname))
    return int(m.group(1)) if m else None


def process_checkpoint(model_path: str,
                       train_df: pd.DataFrame,
                       test_df: pd.DataFrame,
                       train_voxel_dir: str,
                       test_voxel_dir: str,
                       result_path: str,
                       device,
                       add_noise_std_train: float = 0.0):
    """
    """
    model = MultiModalAutoencoder(normalize_shape=True).to(device)
    try:
        state = torch.load(model_path, map_location=device, weights_only=True)
    except TypeError:
        state = torch.load(model_path, map_location=device)
    model.load_state_dict(state, strict=True)

    epoch = parse_epoch_from_filename(model_path)
    if epoch is None:
        stem = os.path.splitext(os.path.basename(model_path))[0]
        train_save = os.path.join(result_path, f"train_embeddings_{stem}.csv")
        test_save  = os.path.join(result_path, f"test_embeddings_{stem}.csv")
        ckpt_label = stem
    else:
        train_save = os.path.join(result_path, f"train_embeddings_epoch_{epoch:03d}.csv")
        test_save  = os.path.join(result_path, f"test_embeddings_epoch_{epoch:03d}.csv")
        ckpt_label = f"{epoch}E"

    with tqdm(total=2,
              desc=f"Checkpoint {ckpt_label}",
              unit="phase") as pbar:

        embed_and_save(
            model, train_df, train_voxel_dir, device, train_save,
            is_test=False, add_noise_std=add_noise_std_train
        )
        pbar.update(1)   # 0% → 50%

        embed_and_save(
            model, test_df, test_voxel_dir, device, test_save,
            is_test=True, add_noise_std=0.0
        )
        pbar.update(1)   # 50% → 100%

# ======================
# ======================

BASE = PROJECT_ROOT

train_voxel_dir = (BASE / "data/voxel_geometry").as_posix()
test_voxel_dir  = train_voxel_dir

##################################################################################################
result_path = (BASE / "data/01_supplier_identification/main_split_70_30").as_posix()
model_dir = (TRAINING_DIR / "checkpoints").as_posix()

train_csv_path = (BASE / "data/01_supplier_identification/main_split_70_30/train_dataset_without_quantity.csv").as_posix()
test_csv_path  = (BASE / "data/01_supplier_identification/main_split_70_30/test_dataset_without_quantity.csv").as_posix()

print("train_voxel_dir:", train_voxel_dir)
print("test_voxel_dir :", test_voxel_dir)
print("result_path    :", result_path)
print("model_dir      :", model_dir)
print("train_csv_path :", train_csv_path)
print("test_csv_path  :", test_csv_path)

# ======================
# ======================

RUN_ALL_CHECKPOINTS = False
RUN_SELECTED_EPOCHS = True

###############################################################################
SELECTED_EPOCHS = [10]

model_path_single = os.path.join(model_dir, "multimodal_autoencoder_epoch_010.pth")
###############################################################################

os.makedirs(result_path, exist_ok=True)

print("[+] Loading train DataFrame from:", train_csv_path)
train_df = load_df(train_csv_path, train_voxel_dir, is_test=False)

print("[+] Loading test DataFrame from:", test_csv_path)
test_df  = load_df(test_csv_path, test_voxel_dir, is_test=True)

print("train_df shape:", train_df.shape)
print("test_df  shape:", test_df.shape)

# ======================
# ======================

if RUN_ALL_CHECKPOINTS:
    pth_files = [
        os.path.join(model_dir, f)
        for f in os.listdir(model_dir)
        if f.endswith(".pth")
    ]

    def sort_key(fp):
        ep = parse_epoch_from_filename(fp)
        return (0, ep) if ep is not None else (1, fp)

    pth_files.sort(key=sort_key)

    if not pth_files:
        print("[!] No .pth files found in:", model_dir)
    else:
        for ckpt in pth_files:
            print(f"\n[>>] Processing checkpoint: {os.path.basename(ckpt)}")
            process_checkpoint(
                ckpt,
                train_df,
                test_df,
                train_voxel_dir,
                test_voxel_dir,
                result_path,
                device,
                add_noise_std_train=0.0,
            )

elif RUN_SELECTED_EPOCHS:
    for ep in SELECTED_EPOCHS:
        ckpt = os.path.join(model_dir, f"multimodal_autoencoder_epoch_{ep:03d}.pth")
        if not os.path.exists(ckpt):
            print(f"[!] checkpoint not found for epoch {ep}: {ckpt}")
            continue

        print(f"\n[>>] Processing checkpoint (selected): {os.path.basename(ckpt)}")
        process_checkpoint(
            ckpt,
            train_df,
            test_df,
            train_voxel_dir,
            test_voxel_dir,
            result_path,
            device,
            add_noise_std_train=0.0,
        )

else:
    if not os.path.exists(model_path_single):
        print("[!] model_path not found:", model_path_single)
    else:
        print(f"[>>] Processing single checkpoint: {os.path.basename(model_path_single)}")
        process_checkpoint(
            model_path_single,
            train_df,
            test_df,
            train_voxel_dir,
            test_voxel_dir,
            result_path,
            device,
            add_noise_std_train=0.0,
        )
